In [2]:

# ============================================================
# SLiCE — STS MODEL
# Output: STS phishing probability
# ============================================================

import numpy as np
import pandas as pd
import joblib

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# 1. Load datasets
legit = pd.read_excel(
    "phishing_email_dataset.xlsx", sheet_name="Dataset"
)
phish = pd.read_excel(
    "phishing_pot_dataset.xlsx", sheet_name="Dataset"
)

df = pd.concat([legit, phish], ignore_index=True)
df = df.drop_duplicates().reset_index(drop=True)

y = df["label"].astype(int)

# 2. Select STS features only
sts_features = [
    c for c in df.columns
    if c.startswith("STS__")
    and c != "STS__new_sender_domain"
    and not c.endswith("_meta")
    and "_meta__" not in c
]

print("STS features:", sts_features)
print("Number of STS features:", len(sts_features))

X = df[sts_features].copy()
X = X.replace([np.inf, -np.inf], np.nan)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.fillna(-1).astype(np.float32)

# 3. Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Balance TRAINING data only: 4,000 legitimate + 4,000 phishing
train_data = X_train.copy()
train_data["label"] = y_train.to_numpy()

legit_train = train_data[train_data["label"] == 0]
phish_train = train_data[train_data["label"] == 1]

phish_sample = phish_train.sample(
    n=len(legit_train),
    random_state=42
)

balanced = pd.concat(
    [legit_train, phish_sample]
).sample(frac=1, random_state=42)

X_balanced = balanced[sts_features]
y_balanced = balanced["label"].astype(int)

print("\nBalanced training distribution:")
print(y_balanced.value_counts())

# 5. Train STS model
model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=60,
    max_depth=6,
    colsample_bytree=0.85,
    subsample=1.0,
    reg_alpha=0,
    reg_lambda=5.0,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(X_balanced, y_balanced)

# 6. Evaluate on the untouched test set
sts_probability = model.predict_proba(X_test)[:, 1]
sts_prediction = (sts_probability >= 0.5).astype(int)

print("\n========== STS MODEL RESULTS ==========")
print("ROC-AUC:", round(
    roc_auc_score(y_test, sts_probability), 6
))
print("PR-AUC:", round(
    average_precision_score(y_test, sts_probability), 6
))
print("\nClassification report:")
print(classification_report(
    y_test,
    sts_prediction,
    target_names=["Legitimate", "Phishing"],
    digits=4
))
print("Confusion matrix:")
print(confusion_matrix(y_test, sts_prediction))

# 7. Save model and its required feature order
joblib.dump(
    {
        "model": model,
        "feature_columns": sts_features,
        "score_name": "STS",
        "output": "phishing_probability",
        "threshold": 0.5
    },
    "slice_sts_model.joblib"
)

print("\nSaved model: slice_sts_model.joblib")

# 8. Example: obtain STS probability for one test email
print("\nExample STS phishing probabilities:")
print(np.round(sts_probability[:10], 4))


STS features: ['STS__dkim_aligned', 'STS__dkim_pass', 'STS__dmarc_pass', 'STS__from_reply_to_mismatch', 'STS__from_return_path_mismatch', 'STS__received_hops', 'STS__sender_domain_entropy', 'STS__spf_pass', 'STS__unusual_received_chain']
Number of STS features: 9

Balanced training distribution:
label
0    4000
1    4000
Name: count, dtype: int64

========== STS MODEL RESULTS ==========
ROC-AUC: 0.999792
PR-AUC: 0.999911

Classification report:
              precision    recall  f1-score   support

  Legitimate     0.9920    0.9960    0.9940      1000
    Phishing     0.9983    0.9967    0.9975      2407

    accuracy                         0.9965      3407
   macro avg     0.9952    0.9963    0.9958      3407
weighted avg     0.9965    0.9965    0.9965      3407

Confusion matrix:
[[ 996    4]
 [   8 2399]]

Saved model: slice_sts_model.joblib

Example STS phishing probabilities:
[9.989e-01 1.000e+00 9.896e-01 1.000e-04 9.999e-01 9.984e-01 1.000e-04
 1.000e-04 7.000e-04 9.995e-01]
